# 🛡️ Drishti Kavach: Training Unified 'RailDrishti' Model on Google Colab (GPU)

This notebook trains the **RailDrishti** (YOLO11-seg) multi-task instance segmentation and obstacle detection model at full **1024×1024 High-Resolution** on a free NVIDIA Tesla T4 / A100 GPU with live minimal **Color-Coded Accuracy Monitoring**.

In [ ]:
# 1. Verify NVIDIA GPU Hardware Acceleration
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install ultralytics -q

In [ ]:
# 3. Mount Google Drive and Extract Dataset
from google.colab import drive
import os, zipfile

drive_zip_path = '/content/drive/MyDrive/raildrishti_colab.zip'
local_zip_path = '/content/raildrishti_colab.zip'

if os.path.exists(local_zip_path):
    print("[+] Found zip in session storage! Extracting...")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("[+] Dataset unzipped successfully into /content/dataset_rail-drishti!")
else:
    try:
        drive.mount('/content/drive')
        if os.path.exists(drive_zip_path):
            print("[+] Found zip in Google Drive! Extracting...")
            with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print("[+] Dataset unzipped successfully from Google Drive!")
        else:
            print("[!] Please upload 'raildrishti_colab.zip' to your Google Drive root folder (MyDrive)!")
    except Exception as e:
        print("[!] Error mounting drive:", e)

In [ ]:
# 4. Generate Colab Dataset Configuration
import yaml

data_yaml = {
    'path': '/content/dataset_rail-drishti',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'Rail_Track_Bed',
        1: 'Rail_Lines',
        2: 'Branch',
        3: 'IronRod',
        4: 'Barrel',
        5: 'Boulder',
        6: 'Jerrycan',
        7: 'Person',
        8: 'Cattle',
        9: 'Animal',
        10: 'Vehicle'
    }
}

with open('/content/raildrishti_colab.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Created /content/raildrishti_colab.yaml")

In [ ]:
# 5. Train Unified 'RailDrishti' Model (YOLO11-seg, 1024 Res, Minimal Color-Coded Monitor)
import warnings, logging, os
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger('ultralytics').setLevel(logging.WARNING)

from ultralytics import YOLO

class EpochStatusMonitor:
    TARGET_BOX_MAP = 85.0
    TARGET_SEG_MAP = 90.0
    TARGET_OVERALL_MAP = 85.0

    def __init__(self):
        self.best_map = 0.0
        self.best_epoch = 0
        self.prev_loss = None
        self.prev_map = None

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        total_epochs = trainer.epochs
        metrics = getattr(trainer, 'metrics', {}) or {}

        box_map50 = (metrics.get('metrics/mAP50(B)', 0.0) or 0.0) * 100.0
        seg_map50 = (metrics.get('metrics/mAP50(M)', 0.0) or 0.0) * 100.0
        overall_map50 = (box_map50 + seg_map50) / 2.0 if (box_map50 > 0 and seg_map50 > 0) else (box_map50 or seg_map50 or 0.0)

        loss_val = None
        if hasattr(trainer, 'tloss') and trainer.tloss is not None:
            try:
                loss_val = float(trainer.tloss.mean()) if hasattr(trainer.tloss, 'mean') else float(trainer.tloss)
            except Exception:
                loss_val = None

        is_new_best = False
        if overall_map50 > self.best_map and overall_map50 > 2.0:
            self.best_map = overall_map50
            self.best_epoch = epoch
            is_new_best = True

        # Color gradient: Red (Worst) -> Green (Best)
        if overall_map50 >= 90.0:
            state_tag = "🏆 [BEST / DEPLOYMENT READY]"
            color = "\033[1;92m"
        elif overall_map50 >= 80.0:
            state_tag = "🌟 [GOOD / OPERATIONAL GRADE]"
            color = "\033[92m"
        elif overall_map50 >= 65.0:
            state_tag = "📈 [APPROACHING TARGET]"
            color = "\033[93m"
        elif overall_map50 >= 45.0:
            state_tag = "🔄 [LEARNING & IMPROVING]"
            color = "\033[95m"
        else:
            state_tag = "🌱 [UNDER-TRAINED / INITIALIZING]"
            color = "\033[91m"

        box_diff = box_map50 - self.TARGET_BOX_MAP
        seg_diff = seg_map50 - self.TARGET_SEG_MAP
        overall_diff = overall_map50 - self.TARGET_OVERALL_MAP

        box_tag = f"[{'+' if box_diff >= 0 else ''}{box_diff:4.1f}%]"
        seg_tag = f"[{'+' if seg_diff >= 0 else ''}{seg_diff:4.1f}%]"
        overall_tag = f"[{'+' if overall_diff >= 0 else ''}{overall_diff:4.1f}%]"

        loss_str = "N/A"
        if loss_val is not None:
            diff_str = ""
            if self.prev_loss is not None:
                d = loss_val - self.prev_loss
                diff_str = f" ({'+' if d > 0 else ''}{d:.3f})"
            loss_str = f"{loss_val:.4f}{diff_str}"

        peak_str = f"{self.best_map:4.1f}% (Ep {self.best_epoch})" + (" *" if is_new_best else "")
        reset = "\033[0m"

        print(f"\n{color}┌─── EPOCH [{epoch:02d}/{total_epochs:02d}] MODEL STATE: {state_tag} ─────────────────────────────┐{reset}")
        print(f"{color}│{reset}  • OVERALL ACCURACY : {color}{overall_map50:5.1f}%{reset} (Target: {self.TARGET_OVERALL_MAP:.1f}% {overall_tag}) | Peak: {peak_str}")
        print(f"{color}│{reset}  • Obstacle Box mAP : {box_map50:5.1f}% / {self.TARGET_BOX_MAP:.1f}% Target {box_tag:<8} | Loss: {loss_str}")
        print(f"{color}│{reset}  • Track Mask mAP   : {seg_map50:5.1f}% / {self.TARGET_SEG_MAP:.1f}% Target {seg_tag:<8} | Status: NOMINAL")
        print(f"{color}└───────────────────────────────────────────────────────────────────────────────┘{reset}\n")

        self.prev_loss = loss_val
        self.prev_map = overall_map50

model = YOLO('yolo11s-seg.pt')
monitor = EpochStatusMonitor()
model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)

results = model.train(
    data='/content/raildrishti_colab.yaml',
    epochs=40,
    imgsz=1024,
    batch=8,
    device=0,
    name='RailDrishti_Training',
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cache=False,
    amp=True,
    save=True,
    patience=20,
    box=7.5,
    cls=1.2,
    dfl=1.8,
    verbose=True
)

In [ ]:
# 6. Save and Download Best Trained Model
from google.colab import files
import os, shutil

best_weight = '/content/runs/segment/RailDrishti_Training/weights/best.pt'
if os.path.exists(best_weight):
    shutil.copy(best_weight, '/content/RailDrishti.pt')
    if os.path.exists('/content/drive/MyDrive'):
        shutil.copy(best_weight, '/content/drive/MyDrive/RailDrishti.pt')
        print("[+] Permanent backup saved to your Google Drive: MyDrive/RailDrishti.pt")
    print("Downloading RailDrishti.pt to your computer...")
    files.download('/content/RailDrishti.pt')
else:
    print("Training not finished or weight not found.")